In [ ]:
# import loompy
#
# files = ["1.Data/Velo/202008.loom",
#          "1.Data/Velo/202201.loom"]
# outputFile = "1.Data/combined.loom"
# loompy.combine(files, outputFile)

In [ ]:
# loading pacakges and settings
import anndata
import scvelo as scv
import pandas as pd
import scanpy as sc
# import cellrank as cr
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['savefig.facecolor']='#ffffff'
plt.rcParams['figure.facecolor']='#ffffff'

In [ ]:
# readin the loom file and check cell IDs
sample = anndata.read_loom("combined.loom")
sample.obs.head

In [ ]:
sample.obs=sample.obs.rename(index = lambda x: x.replace('count:', 'nor_').replace('LA_5_Gex:', 'pml_').replace('x', ''))
sample.obs.head

In [ ]:
# Load information from Seurat project
sample_obs = pd.read_csv("Normal CD8_cellID.csv")
umap = pd.read_csv("Normal CD8_umap.csv")
umap = umap.rename(columns = {"Unnamed: 0":'CellID'})
cell_clusters = pd.read_csv("Normal CD8_clusters.csv")
cell_clusters = cell_clusters.rename(columns = {"Unnamed: 0":'CellID'})
cell_clusters.head

In [ ]:
# Combine seurat and volecity 
sample_one = sample[np.isin(sample.obs.index,sample_obs["x"])]
sample_one.obs.head()
sample_one_index = pd.DataFrame(sample_one.obs.index)
sample_one_index = sample_one_index.rename(columns = {0:'CellID'})
umap_ordered = sample_one_index.merge(umap,on="CellID")
cell_clusters = sample_one_index.merge(cell_clusters,on="CellID")
umap_ordered = umap_ordered.iloc[:,1:]
sample_one.obsm['X_umap'] = umap_ordered.values
cell_clusters_ordered=cell_clusters.iloc[:,1]
sample_one.obs['cell_clusters']=cell_clusters_ordered.values

In [ ]:
# ident_colours = pd.read_csv('col.csv')
col = pd.read_csv('Normal CD8_col.csv').loc[:,'col']
ident_colours = col.tolist()

coordDf = pd.read_csv('Normal CD8_coord.csv')
figSize = [coordDf['x'].iloc[0]/2, coordDf['y'].iloc[0]/2]

scv.set_figure_params(style='scvelo', dpi=300, dpi_save=300, 
                         frameon=None, vector_friendly=True, 
                         transparent=False, fontsize=12, 
                         figsize=None, color_map=None, 
                         facecolor=None, format='png', 
                      ipython_format='png2x')

In [ ]:
#Running RNA Velocity and ploting
scv.pp.filter_and_normalize(sample_one,min_shared_counts=20, n_top_genes=2000)
# scv.pp.neighbors(sample_one)
scv.pp.moments(sample_one, n_pcs=30, n_neighbors=30)
scv.tl.recover_dynamics(sample_one, n_jobs=1)
scv.tl.velocity(sample_one, mode = "stochastic") # stochastic dynamical deterministic
scv.tl.velocity_graph(sample_one)

In [ ]:
scv.pl.velocity_embedding_stream(sample_one,
                                 basis='X_umap',
                                 arrow_style="<|-",
                                 color = "cell_clusters",
                                 palette = ident_colours, 
                                 smooth = 1, 
                                 min_mass = 0.8,
                                 density = 1.2,
                                 legend_loc='none', 
                                 title="",
                                 size=200,
                                 # alpha=0.5,
                                 figsize= figSize,
                                 save='plcCellVelo.png',
                                 dpi = 300)

In [ ]:
scv.pl.proportions(sample_one, groupby='cell_clusters',
                   # figsize=[6,12],
                   save='plcSplicedDistr.png',
                   dpi = 300)

In [ ]:
sample_one.uns['neighbors']['distances'] = sample_one.obsp['distances']
sample_one.uns['neighbors']['connectivities'] = sample_one.obsp['connectivities']

scv.tl.paga(sample_one, groups='cell_clusters')
df = scv.get_df(sample_one, 'paga/transitions_confidence', precision=2).T
df.style.background_gradient(cmap='Blues').format('{:.2g}')

In [ ]:
scv.pl.paga(sample_one, 
            basis='umap', 
            size=50,
            # threshold=0.5, 
            alpha=.3,
            layout = "fa",
            min_edge_width=2, 
            figsize= figSize,
            title="",
            save='pMELCellVeloPaga.png',
            node_size_scale=1.5)

In [ ]:
ident_colours = ["#F8766D", "#9590FF"]

scv.pl.velocity_embedding_grid(sample_one[sample_one.obs['cell_clusters'].isin(['0', '7'])],
                               basis='umap',
                               color='cell_clusters',
                               palette = ident_colours,
                               save='embeddingGridSubset.png',
                               figsize=[5, 8.1],
                               density = 0.5,
                               min_mass = 50,
                               smooth = 0.5,
                               title='',
                               scale=0.25,
                               dpi = 300)

scv.pl.velocity_embedding_stream(sample_one[sample_one.obs['cell_clusters'].isin(['0', '7'])],
                                 basis='X_umap',
                                 color = "cell_clusters",
                                 palette = ident_colours,
                                 # legend_loc='none',
                                 title="",
                                 min_mass = 20,
                                 size=200,
                                 # alpha=0.5,
                                 figsize=[5, 8.1],
                                 save='velocitySubset.png',
                                 dpi = 300)

In [ ]:
# Identify important genes
scv.tl.rank_velocity_genes(sample_one, groupby='cell_clusters', min_corr=.5)

df = scv.DataFrame(sample_one.uns['rank_velocity_genes']['names'])
df.to_csv("keyFeaturesbyCluster.csv")
df.head(10)

In [ ]:
sample_one.obs.to_csv("newMeta.csv")

In [ ]:
kwargs = dict(frameon=False, size=10, linewidth=1.5,
              add_outline='1, 2, 3')

scv.pl.scatter(sample_one, df['1'][:5], ylabel='1', frameon=False, color='cell_clusters',
               size=10, linewidth=1.5)
# scv.pl.scatter(sample_one, df['Lef1 high EP'][:5], ylabel='Lef1 high EP', **kwargs)
# scv.pl.scatter(sample_one, df['Pre-endocrine'][:5], ylabel='Pre-endocrine', **kwargs)

In [ ]:
scv.tl.velocity_confidence(sample_one)
keys = 'velocity_length', 'velocity_confidence'
scv.pl.scatter(sample_one, c=keys, cmap='coolwarm', perc=[5, 95])

In [ ]:
scv.pl.velocity_graph(sample_one, threshold=1, palette = ident_colours, color='cell_clusters')

In [ ]:
x, y = scv.utils.get_cell_transitions(sample_one, basis='umap', starting_cell=50)
ax = scv.pl.velocity_graph(sample_one, c='lightgrey', edge_width=.05, show=False)
ax = scv.pl.scatter(sample_one, x=x, y=y, s=120, c='ascending', cmap='gnuplot', ax=ax)

In [ ]:
scv.tl.velocity_pseudotime(sample_one)
scv.pl.scatter(sample_one, color='velocity_pseudotime', cmap='gnuplot',
               title="",
               size=50,
               # alpha=0.5,
               figsize=[6,6],
               save='velocityPseudotime.png',)

In [ ]:
sample_one.uns['neighbors']['distances'] = sample_one.obsp['distances']
sample_one.uns['neighbors']['connectivities'] = sample_one.obsp['connectivities']
scv.tl.paga(sample_one, groups='cell_clusters')
df = scv.get_df(sample_one, 'paga/transitions_confidence', precision=2).T
df.style.background_gradient(cmap='Blues').format('{:.2g}')

In [ ]:
scv.pl.paga(sample_one, basis='umap', 
            size=50, alpha=.2,
            min_edge_width=2, 
            node_size_scale=1.5, 
            title="",
            # size=100,
            figsize=[6,6],
            save='developPath.png',
            dpi = 300)

In [ ]:
scv.tl.recover_dynamics(sample_one)

In [ ]:
scv.tl.velocity(sample_one, mode='dynamical')

In [ ]:
scv.tl.velocity_graph(sample_one)

In [ ]:
df = sample_one.var
df = df[(df['fit_likelihood'] > .1) & df['velocity_genes'] == True]

In [ ]:
kwargs = dict(xscale='log', fontsize=16)

In [ ]:
with scv.GridSpec(ncols=3) as pl:
    pl.hist(df['fit_alpha'], xlabel='transcription rate', **kwargs)
    pl.hist(df['fit_beta'] * df['fit_scaling'], xlabel='splicing rate', xticks=[.1, .4, 1], **kwargs)
    pl.hist(df['fit_gamma'], xlabel='degradation rate', xticks=[.1, .4, 1], **kwargs)

In [ ]:
scv.get_df(sample_one, 'fit*', dropna=True).head()

In [ ]:
scv.tl.latent_time(sample_one)

In [ ]:
scv.pl.scatter(sample_one, 
               color='latent_time', 
               color_map='gnuplot', 
               size=80, 
               save='latentTime.png',
               # alpha=0.5,
               figsize=[5,5],
               title="")

In [ ]:
sc.pl.violin(sample_one,
             keys='latent_time',
             groupby="cell_clusters")

In [ ]:
top_genes = sample_one.var['fit_likelihood'].sort_values(ascending=False).index[:1000]
scv.pl.heatmap(sample_one,
               var_names=top_genes,
               # sortby='cell_clusters',
               col_color='cell_clusters',
               n_convolve=300,
               save='topGHeatmap.png',
               # alpha=0.5,
               font_scale = 0.25,
               figsize=[8, 8])

In [ ]:
scv.pl.heatmap(sample_one[sample_one.obs["cell_clusters"].isin(['5'])],
               var_names=top_genes,
               # sortby='cell_clusters',
               col_color='cell_clusters',
               n_convolve=300,
               save='topGHeatmapH5.png',
               # alpha=0.5,
               font_scale = 0.25,
               figsize=[5, 0.78])

In [ ]:
sc.pl.heatmap(sample_one, n_genes=3, standard_scale='var')

In [ ]:
top_genes

In [ ]:
top_genes = sample_one.var['fit_likelihood'].sort_values(ascending=False).index
scv.pl.scatter(sample_one, color='cell_clusters', basis=top_genes[:15], ncols=5, frameon=False)

In [ ]:
var_names = ['Gm20481', 'Malat1', 'Lef1', 'Macf1', 'Dgka']
scv.pl.scatter(sample_one, var_names, 
               color='cell_clusters', 
               frameon=False, 
              save='selectGs.png')
scv.pl.scatter(sample_one, x='latent_time', y=var_names, 
               color='cell_clusters', frameon=False, 
               save='selectGs2.png')